# SAC Irrigation Training — v2.7.0 (Colab Pro)

**Architecture:** CTDE + VDN Factorized Critic  
**Hyperparameters:** `ent_coef=0.05` (fixed), `max_grad_norm=1.0`, LR decay

## Key changes from v2.6
| # | Change | Effect |
|---|--------|--------|
| 1 | gamma obs slot restored to `elev_norm` | Bug fix: was writing field-uniform GDD scalar instead of per-agent elevation |
| 2 | 3 new static topo features per agent | Actor can now differentiate agents by cascade role, not just soil moisture |
| 3 | OBS_DIM 707 → 1097 | Per-agent block grows from 5 to 8 features |
| 4 | Episodes always run 93 days | Agent now feels late-season drought after overspending; no early termination |
| 5 | Reward simplified to r1+r2+r3+r6 | Burn-rate `rb` and dead delta-u `r5` removed |

## Before running
1. `Runtime → Change runtime type → A100 GPU` (Colab Pro). If A100 is unavailable, L4 is the next best choice.
2. Add WandB API key as a Colab Secret: left sidebar → key icon → `+ Add new secret` → Name: `WANDB_API_KEY` → toggle **Notebook access ON**.
3. Mount Google Drive in Cell 1 — required for results to survive session disconnects.

## What to watch on WandB
| Metric | Expected v2.7 behaviour | Abort if |
|--------|------------------------|----------|
| `train/critic_loss` | Falls from ~2700 to <50 by step 25k, stays bounded | Exceeds 500 after step 50k |
| `train/ent_coef` | **Constant at 0.05 throughout** | Deviates — means wrong SB3 version |
| `rollout/ep_len_mean` | Reaches **exactly 93** and stays there | Below 93 at step 25k (termination bug) |
| `rollout/ep_rew_mean` | More negative early than v2.6 (full-season drought accumulation is normal); improves over time | Monotonically decreasing after step 100k |
| GPU utilization | 40–70% (env is CPU-bound) | <10% (GPU idle — env bottleneck) |

## Pilot run (mandatory before full 500k)
Run Cell 4 with `TOTAL_STEPS = 25_000` first. If `critic_loss < 50` at step 25k and `ep_len_mean == 93`, change to `500_000` and continue in the same session.

## Seed plan
One seed per Colab session. Change `SEED` in Cell 4.
- Session 1 → `SEED = 0`
- Session 2 → `SEED = 1`
- Session 3 → `SEED = 2`
- Session 4 → `SEED = 3`
- Session 5 → `SEED = 4`

Results saved to `MyDrive/thesis_results/sac_seed{N}_v27_{timestamp}/`.

## Estimated runtimes (v2.7 obs is ~10% larger than v2.6)
| Hardware | Steps/sec | 500k steps |
|----------|-----------|------------|
| A100 | ~120–160 | ~55–70 min |
| L4 | ~75–100 | ~85–110 min |
| T4 | ~55–75 | ~110–150 min |

In [ ]:
# ── CELL 1: Mount Drive, clone repo, install deps ───────────────────────────
import subprocess, sys, os

from google.colab import drive
drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT = '/content/drive/MyDrive/thesis_results'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f'✓  Drive mounted. Results → {DRIVE_ROOT}')

if os.path.exists('/content/thesis'):
    subprocess.run(['rm', '-rf', '/content/thesis'], check=True)
subprocess.run(
    ['git', 'clone', 'https://github.com/taratorbati/thesis.git', '/content/thesis'],
    check=True
)
os.chdir('/content/thesis')
sys.path.insert(0, '/content/thesis')
print('✓  Repo cloned')

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'stable-baselines3==2.6.0',
    'gymnasium==1.1.1',
    'wandb>=0.16',
    'pytest',
], check=True)

import numpy as np, gymnasium, stable_baselines3 as sb3, torch
print(f'numpy:             {np.__version__}')
print(f'gymnasium:         {gymnasium.__version__}')
print(f'stable-baselines3: {sb3.__version__}')
print(f'torch:             {torch.__version__}')
print(f'CUDA available:    {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:               {torch.cuda.get_device_name(0)}')
torch.set_num_threads(4)

In [ ]:
# ── CELL 2: WandB secret + GPU check ────────────────────────────────────────
import os
try:
    from google.colab import userdata
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    print('✓  WANDB_API_KEY loaded from Colab Secrets.')
except Exception as e:
    print(f'⚠  Could not load WANDB_API_KEY ({type(e).__name__}).')
    print('   Training continues without WandB — add key to Colab Secrets to enable.')

import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else 'nvidia-smi failed — no GPU allocated')

In [ ]:
# ── CELL 3: Pre-training validation (MUST PASS before Cell 4) ───────────────
#
# Runs smoke tests + VDN unit tests + step-rate benchmark.
# DO NOT proceed to Cell 4 if any test fails.

import subprocess, sys, time

print('Running smoke tests...')
r = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_rl_smoke.py', '-v', '--tb=short'],
    capture_output=False
)
assert r.returncode == 0, 'SMOKE TESTS FAILED — check output above'
print()

print('Running VDN unit tests...')
r2 = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_factorized_critic.py', '-v', '--tb=short'],
    capture_output=False
)
assert r2.returncode == 0, 'VDN UNIT TESTS FAILED — check output above'
print()

print('Benchmarking step rate...')
from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import DummyVecEnv
from src.rl.gym_env import IrrigationEnv
from src.rl.networks import CTDESACPolicy, make_sac_policy_kwargs

bench_env = DummyVecEnv([lambda: IrrigationEnv(randomize=True)])
bench_model = SAC(
    policy=CTDESACPolicy, env=bench_env,
    policy_kwargs=make_sac_policy_kwargs(N=130),
    buffer_size=5_000, batch_size=256, learning_starts=500,
    ent_coef=0.05, verbose=0, seed=0,
)
bench_model.learn(total_timesteps=300)
t0 = time.time()
bench_model.learn(total_timesteps=500, reset_num_timesteps=False)
rate = 500 / (time.time() - t0)
del bench_model, bench_env

print(f'Step rate:    {rate:.1f} steps/sec')
print(f'Est. 500k:    {500_000 / rate / 60:.0f} min  ({500_000 / rate / 3600:.1f} h)')
print()

from src.rl.gym_env import IrrigationEnv, OBS_DIM
env_check = IrrigationEnv(randomize=False)
obs_check, _ = env_check.reset()
assert obs_check.shape[0] == 1097, f'Expected obs_dim=1097, got {obs_check.shape[0]}'
print(f'obs_dim:      {obs_check.shape[0]} ✓  (8 features × 130 agents + 9 scalars + 48 forecast)')
print()
print('✓  ALL VALIDATION PASSED — safe to proceed to Cell 4')

In [ ]:
# ── CELL 4: Training ─────────────────────────────────────────────────────────
#
# PILOT RUN FIRST — mandatory:
#   Set TOTAL_STEPS = 25_000 and run. Check WandB:
#     - critic_loss < 50 at step 25k
#     - ep_len_mean == 93 (not 83 — full-season episodes)
#   If both pass, change TOTAL_STEPS to 500_000 and re-run this cell.
#
# Change SEED for each parallel session (0, 1, 2, 3, 4).

SEED = 0   # ← CHANGE THIS for each session
TOTAL_STEPS = 500_000   # ← set to 25_000 for mandatory pilot; 500_000 for full run

from src.rl.train import train_sac

model = train_sac(
    seed=SEED,
    output_dir='/content/thesis/results/rl',
    wandb_project='sac-irrigation-thesis',
    total_timesteps=TOTAL_STEPS,
)
print(f'\n✓  Training complete — seed {SEED}, steps {TOTAL_STEPS}')

In [ ]:
# ── CELL 5: Copy results to Google Drive ────────────────────────────────────
# Run immediately after Cell 4 finishes.
# Copies everything EXCEPT the replay buffer (multi-GB, not needed for eval).

import shutil, os, datetime

src = f'/content/thesis/results/rl/sac_general_seed{SEED}'
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
dst = f'{DRIVE_ROOT}/sac_seed{SEED}_v27_{timestamp}'

def ignore_replay_buffer(directory, files):
    return [f for f in files if 'replay_buffer' in f]

shutil.copytree(src, dst, ignore=ignore_replay_buffer)

total_mb, file_count = 0, 0
for root, dirs, files in os.walk(dst):
    for f in files:
        size = os.path.getsize(os.path.join(root, f))
        total_mb += size / 1e6
        file_count += 1
        rel = os.path.relpath(os.path.join(root, f), dst)
        print(f'  {rel}  ({size/1e6:.2f} MB)')

print(f'\n✓  {file_count} files ({total_mb:.1f} MB) saved to:')
print(f'   {dst}')

In [ ]:
# ── CELL 6: Quick post-training diagnostics ──────────────────────────────────
#
# Loads best_model and checks the three key v2.7 targets:
#   spatial_std   > 0.01  (v2.6 was ~0.002 — spatially blind)
#   corr(u, rain) < 0     (v2.6 was +0.38 — irrigated MORE when raining)
#   corr(u, x1)   < 0     (v2.6 was +0.03 — no closed-loop response)
#   ep_len        == 93   (v2.6 was ~83 due to early termination)

import numpy as np
from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import DummyVecEnv
from src.rl.gym_env import IrrigationEnv

best_model_path = f'/content/thesis/results/rl/sac_general_seed{SEED}/best_model/best_model'

eval_env = DummyVecEnv([lambda: IrrigationEnv(randomize=False)])
loaded = SAC.load(best_model_path, env=eval_env)

obs = eval_env.reset()
all_actions, all_rain, all_x1 = [], [], []
ep_len = 0
done = False
while not done:
    action, _ = loaded.predict(obs, deterministic=True)
    all_actions.append(action[0].copy())
    raw_obs = obs[0]
    # x1 mean across agents from agent-major obs block (feature index 0 of 8)
    x1_vals = raw_obs[:1040].reshape(130, 8)[:, 0]
    all_x1.append(x1_vals.mean())
    # rain is scalar block position 1044 (per-agent 1040 + scalar[4])
    all_rain.append(float(raw_obs[1044]))
    obs, reward, done, info = eval_env.step(action)
    ep_len += 1
    done = done[0]

all_actions = np.array(all_actions)
u_arr      = all_actions.mean(axis=1) * 12
spatial_std = all_actions.mean(axis=0).std()
corr_rain   = float(np.corrcoef(u_arr, np.array(all_rain))[0, 1])
corr_x1     = float(np.corrcoef(u_arr, np.array(all_x1))[0, 1])

print('=== v2.7 Post-training Diagnostics ===')
print(f'Episode length:         {ep_len}  (expected 93)')
print(f'Mean irrigation mm/day: {u_arr.mean():.2f}')
print(f'Spatial std:            {spatial_std:.4f}  (target > 0.01; v2.6 was ~0.002)')
print(f'corr(u, rain_today):    {corr_rain:+.3f}  (target < 0; v2.6 was +0.38)')
print(f'corr(u, x1_mean):       {corr_x1:+.3f}  (target < 0; v2.6 was +0.03)')
print()
checks = [
    (ep_len == 93,       f'ep_len {ep_len} == 93'),
    (spatial_std > 0.01, f'spatial_std {spatial_std:.4f} > 0.01'),
    (corr_rain < 0,      f'corr(u,rain) {corr_rain:+.3f} < 0'),
    (corr_x1   < 0,      f'corr(u,x1)   {corr_x1:+.3f} < 0'),
]
for ok, label in checks:
    print(f'  {"✓" if ok else "⚠"} {label}')

In [ ]:
# ── CELL 7: Resume from Drive checkpoint (if session was interrupted) ────────
# If the session disconnected mid-training, resume from the latest checkpoint.
# Fill in CHECKPOINT_STEP and CHECKPOINT_DRIVE_PATH, then uncomment and run.

# SEED = 0
# CHECKPOINT_STEP = 200_000
# CHECKPOINT_DRIVE_PATH = f'{DRIVE_ROOT}/sac_seed{SEED}_v27_YYYYMMDD_HHMMSS'
#
# import shutil, os
# local_dir = f'/content/thesis/results/rl/sac_general_seed{SEED}'
# os.makedirs(local_dir, exist_ok=True)
# shutil.copytree(CHECKPOINT_DRIVE_PATH, local_dir, dirs_exist_ok=True)
#
# from stable_baselines3 import SAC
# from stable_baselines3.common.vec_env import DummyVecEnv
# from src.rl.gym_env import IrrigationEnv
# from src.rl.train import _make_lr_schedule, LR_START, LR_END, TOTAL_TIMESTEPS
#
# env = DummyVecEnv([lambda: IrrigationEnv(randomize=True)])
# ckpt_zip = f'{local_dir}/checkpoints/sac_general_seed{SEED}_{CHECKPOINT_STEP}_steps'
# model = SAC.load(ckpt_zip, env=env)
# model.lr_schedule = _make_lr_schedule(LR_START, LR_END)
#
# remaining = TOTAL_TIMESTEPS - CHECKPOINT_STEP
# print(f'Resuming from step {CHECKPOINT_STEP}, {remaining} steps remaining...')
# model.learn(total_timesteps=remaining, reset_num_timesteps=False, progress_bar=True)
# model.save(f'{local_dir}/sac_general_seed{SEED}_final')
# print('Resume complete.')